# Rainbow / Multi-Spread Option - Model Comparison (Presentation vs Li-Zhou-Deng vs Monte Carlo)

This notebook implements and compares three approaches for valuing the option to divert among multiple destinations (e.g., regas terminals) when the payoff can be represented as max_k (S_k - S_1)^+.

Models:
1) Presentation approximation (pairwise Kirk/Margrabe) with outranking probabilities.
2) Li-Zhou-Deng (LZD) conditional quadrature per pair, same outranking scheme.
3) Monte Carlo simulation with correlated GBMs.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numpy.polynomial.hermite import hermgauss
from time import perf_counter
from dataclasses import dataclass
from typing import Tuple
from scipy.stats import norm
plt.style.use('seaborn-v0_8')
np.random.seed(42)

In [ ]:
def black_call(F, K, vol_sqrtT):
    F = float(F); K = float(K)
    if vol_sqrtT <= 1e-12:
        return max(F - K, 0.0)
    d1 = (np.log(F / K) + 0.5 * vol_sqrtT**2) / vol_sqrtT
    d2 = d1 - vol_sqrtT
    return F * norm.cdf(d1) - K * norm.cdf(d2)

def margrabe(F1, F2, sigma1, sigma2, rho, T):
    v = (sigma1**2 + sigma2**2 - 2*rho*sigma1*sigma2) * T
    vs = np.sqrt(max(v, 0.0))
    if vs <= 1e-14:
        return max(F2 - F1, 0.0)
    d1 = (np.log(F2/F1) + 0.5*v) / vs
    d2 = d1 - vs
    return F2 * norm.cdf(d1) - F1 * norm.cdf(d2)

def kirk(F1, F2, sigma1, sigma2, rho, T, K=0.0):
    if abs(K) < 1e-15:
        return margrabe(F1, F2, sigma1, sigma2, rho, T)
    beta = F2 / (F2 + K)
    v = (sigma1**2 - 2*beta*rho*sigma1*sigma2 + (beta**2)*sigma2**2) * T
    vs = np.sqrt(max(v, 0.0))
    d1 = (np.log(F1/(F2+K)) + 0.5*v) / vs
    d2 = d1 - vs
    return F1 * norm.cdf(d1) - (F2 + K) * norm.cdf(d2)

def lzd_conditional(F1, F2, sigma1, sigma2, rho, T, K=0.0, n=32):
    z, w = hermgauss(n)
    mu1 = np.log(F1) - 0.5 * (sigma1**2) * T
    s1 = sigma1 * np.sqrt(T)
    v2_cond = (1 - rho**2) * sigma2**2 * T
    total = 0.0
    for zi, wi in zip(z, w):
        lnS1 = mu1 + np.sqrt(2.0) * s1 * zi
        S1 = np.exp(lnS1)
        mu2_given1 = (np.log(F2) - 0.5*sigma2**2*T) + (rho * sigma2 / sigma1) * (lnS1 - mu1)
        F2_cond = np.exp(mu2_given1 + 0.5 * v2_cond)
        K_eff = S1 + K
        if K_eff <= 1e-15:
            price_cond = F2_cond
        else:
            price_cond = black_call(F2_cond, K_eff, np.sqrt(v2_cond))
        total += wi * price_cond
    return float(total / np.sqrt(np.pi))

def outrank_prob(Fi, Fj, si, sj, rho_ij, T):
    mu = (np.log(Fi) - 0.5*si**2*T) - (np.log(Fj) - 0.5*sj**2*T)
    var = (si**2 + sj**2 - 2*rho_ij*si*sj) * T
    s = np.sqrt(max(var, 1e-18))
    return float(norm.cdf(mu / s))

@dataclass
class Market:
    F: np.ndarray
    sigma: np.ndarray
    rho: np.ndarray
    T: float = 1.0
    K: float = 0.0
    r: float = 0.0
    def check(self):
        m = len(self.F)
        assert self.sigma.shape == (m,), 'sigma shape mismatch'
        assert self.rho.shape == (m,m), 'rho shape mismatch'
        ev = np.linalg.eigvalsh(self.rho)
        if ev.min() < -1e-6:
            raise ValueError('Correlation matrix not PSD')

In [ ]:
def presentation_price(mkt: Market):
    mkt.check()
    F, s, R, T, K = mkt.F, mkt.sigma, mkt.rho, mkt.T, mkt.K
    base = 0
    alts = np.argsort(-F[1:]) + 1
    terms = []
    def spread(i):
        return kirk(F[base], F[i], s[base], s[i], R[base, i], T, K)
    price = 0.0
    if len(alts) >= 1:
        p2 = spread(alts[0]); price += p2; terms.append((int(alts[0]+1), p2, 1.0))
    if len(alts) >= 2:
        i3 = alts[1]; p3 = spread(i3)
        P32 = outrank_prob(F[i3], F[alts[0]], s[i3], s[alts[0]], R[i3, alts[0]], T)
        price += p3 * P32; terms.append((int(i3+1), p3, P32))
    if len(alts) >= 3:
        i4 = alts[2]; p4 = spread(i4)
        P42 = outrank_prob(F[i4], F[alts[0]], s[i4], s[alts[0]], R[i4, alts[0]], T)
        P43 = outrank_prob(F[i4], F[alts[1]], s[i4], s[alts[1]], R[i4, alts[1]], T)
        price += p4 * P42 * P43; terms.append((int(i4+1), p4, P42*P43))
    return price, terms

def lzd_ext_price(mkt: Market, n_gh: int = 32):
    mkt.check()
    F, s, R, T, K = mkt.F, mkt.sigma, mkt.rho, mkt.T, mkt.K
    base = 0
    alts = np.argsort(-F[1:]) + 1
    def spread_lzd(i):
        return lzd_conditional(F[base], F[i], s[base], s[i], R[base, i], T, K, n=n_gh)
    price = 0.0; terms = []
    if len(alts) >= 1:
        p2 = spread_lzd(alts[0]); price += p2; terms.append((int(alts[0]+1), p2, 1.0))
    if len(alts) >= 2:
        i3 = alts[1]; p3 = spread_lzd(i3)
        P32 = outrank_prob(F[i3], F[alts[0]], s[i3], s[alts[0]], R[i3, alts[0]], T)
        price += p3 * P32; terms.append((int(i3+1), p3, P32))
    if len(alts) >= 3:
        i4 = alts[2]; p4 = spread_lzd(i4)
        P42 = outrank_prob(F[i4], F[alts[0]], s[i4], s[alts[0]], R[i4, alts[0]], T)
        P43 = outrank_prob(F[i4], F[alts[1]], s[i4], s[alts[1]], R[i4, alts[1]], T)
        price += p4 * P42 * P43; terms.append((int(i4+1), p4, P42*P43))
    return price, terms

def mc_price(mkt: Market, n_sims: int = 100000, antithetic: bool = True, seed: int = 7):
    mkt.check()
    F, s, R, T, K = mkt.F, mkt.sigma, mkt.rho, mkt.T, mkt.K
    rng = np.random.default_rng(seed)
    m = len(F)
    L = np.linalg.cholesky(R + 1e-12*np.eye(m))
    n = n_sims
    if antithetic:
        n = (n_sims + 1)//2
    Z = rng.standard_normal((n, m))
    if antithetic:
        Z = np.vstack([Z, -Z])
    Z = Z[:n_sims]
    shocks = Z @ L.T
    drift = (-0.5 * (s**2) * T)
    vol = s * np.sqrt(T)
    lnS = np.log(F) + drift + shocks * vol
    S = np.exp(lnS)
    base = S[:, 0]
    alts = S[:, 1:]
    payoff = np.maximum(alts - base[:, None] - K, 0.0)
    best = payoff.max(axis=1)
    return best.mean()

## 4. Test Scenarios and Experiment Runner
We evaluate both 3-asset and 4-asset setups under a moderate-vol / moderate-corr regime.

In [ ]:
F3 = (50.0, 52.0, 51.0)
sig3 = (0.35, 0.30, 0.30)
R3 = np.array([[1.0, 0.6, 0.5], [0.6, 1.0, 0.55], [0.5, 0.55, 1.0]])
F4 = (50.0, 52.0, 51.0, 49.5)
sig4 = (0.35, 0.30, 0.30, 0.32)
R4 = np.array([[1.0, 0.6, 0.5, 0.55], [0.6, 1.0, 0.55, 0.5], [0.5, 0.55, 1.0, 0.6], [0.55, 0.5, 0.6, 1.0]])
scenarios = [('3-assets', F3, sig3, R3), ('4-assets', F4, sig4, R4)]
T = 0.5
K = 0.0
results = []
for name, F, sig, R in scenarios:
    mkt = Market(F=np.array(F, dtype=float), sigma=np.array(sig, dtype=float), rho=R.astype(float), T=T, K=K)
    t0 = perf_counter(); v_pres, terms_pres = presentation_price(mkt); t1 = perf_counter()
    t2 = perf_counter(); v_lzd,  terms_lzd  = lzd_ext_price(mkt, n_gh=32); t3 = perf_counter()
    t4 = perf_counter(); v_mc = mc_price(mkt, n_sims=200000, antithetic=True, seed=123); t5 = perf_counter()
    results.append({'Case': name, 'Presentation_Value': v_pres, 'LZDext_Value': v_lzd, 'MonteCarlo_Value': v_mc,
                    'Presentation_Time_ms': (t1-t0)*1e3, 'LZDext_Time_ms': (t3-t2)*1e3, 'MonteCarlo_Time_ms': (t5-t4)*1e3,
                    'Pres_Terms': terms_pres, 'LZD_Terms': terms_lzd})
res = pd.DataFrame(results)
res

## 5. Plots: Value & Runtime

In [ ]:
for case in res['Case']:
    row = res[res['Case']==case].iloc[0]
    vals = [row['Presentation_Value'], row['LZDext_Value'], row['MonteCarlo_Value']]
    labs = ['Presentation', 'LZD-ext', 'Monte-Carlo']
    times = [row['Presentation_Time_ms'], row['LZDext_Time_ms'], row['MonteCarlo_Time_ms']]
    fig, axes = plt.subplots(1, 2, figsize=(10,4))
    axes[0].bar(labs, vals, color=['#2a9d8f','#264653','#e76f51'])
    axes[0].set_title(f'{case}: Values')
    axes[0].set_ylabel('Option Value')
    axes[0].grid(True, axis='y', alpha=0.3)
    axes[1].bar(labs, times, color=['#2a9d8f','#264653','#e76f51'])
    axes[1].set_title(f'{case}: Runtime (ms)')
    axes[1].set_ylabel('Milliseconds')
    axes[1].grid(True, axis='y', alpha=0.3)
    plt.tight_layout(); plt.show()

## 6. Applicability Stress: Error vs Correlation

In [ ]:
corr_levels = np.linspace(0.0, 0.9, 10)
errs_pres, errs_lzd = [], []
for c in corr_levels:
    R = np.array([[1.0, c, c], [c, 1.0, 0.5], [c, 0.5, 1.0]])
    mkt = Market(F=np.array(F3), sigma=np.array(sig3), rho=R, T=T, K=K)
    vp,_ = presentation_price(mkt)
    vl,_ = lzd_ext_price(mkt)
    vm = mc_price(mkt, n_sims=120000, antithetic=True, seed=321)
    errs_pres.append(abs(vp - vm))
    errs_lzd.append(abs(vl - vm))
plt.figure(figsize=(6,4))
plt.plot(corr_levels, errs_pres, label='Presentation abs error')
plt.plot(corr_levels, errs_lzd, label='LZD-ext abs error')
plt.xlabel('Corr(base, alt)')
plt.ylabel('Absolute error vs MC')
plt.title('3-asset: Error vs correlation')
plt.grid(True, alpha=0.3)
plt.legend(); plt.show()

## 7. Tabular Summary

In [ ]:
disp = res[['Case','Presentation_Value','LZDext_Value','MonteCarlo_Value','Presentation_Time_ms','LZDext_Time_ms','MonteCarlo_Time_ms']].copy()
disp['Pres_vs_MC_%'] = 100.0*(disp['Presentation_Value']/disp['MonteCarlo_Value'] - 1)
disp['LZD_vs_MC_%'] = 100.0*(disp['LZDext_Value']/disp['MonteCarlo_Value'] - 1)
disp.round(4)

## 8. Notes
- Presentation: pairs via Kirk (Margrabe when K=0) times outranking probabilities.
- LZD-ext: 1-D Gauss-Hermite per pair with same outranking scheme.
- Monte Carlo: correlated GBMs, antithetic, Cholesky.
Caveats: outranking factorization is a simplification; high correlation or close forwards can bias the approximations.